# Privacy d'export & pattern mining — `opaque_id`, `sanitize_state`, `pattern_mining`

Sixième volet de l'épic #1961 (Phase 6) : la **chaîne d'export privacy** du
dépôt et le **minage de motifs** agrégés. Tout ce qui suit rejoue les modules
réels — Python pur, zéro JVM, zéro LLM, **zéro corpus** (fixtures synthétiques
uniquement, la discipline est le sujet du notebook) — et chaque cellule
vérifie ses résultats contre `privacy_mining_examples.json`.

## Les trois fichiers

| Module | Lignes | Rôle |
|---|---|---|
| `evaluation/opaque_id.py` | 65 | ID opaques déterministes sha256(salt+name) — sel secret obligatoire |
| `evaluation/sanitize_state.py` | 658 | LE garde d'export : allowlist, agrégats préservés, texte nominatif retiré |
| `evaluation/pattern_mining.py` | 511 | Spectres, asymétrie Tricherie/Influence, co-occurrence, couverture croisée |

## Le fil : que peut quitter la machine ?

Le dataset canonique est chiffré ; les états de travail portent du texte réel.
`sanitize_state` est l'unique point de passage avant git/dashboard/PR :
**allowlist-driven** — ce qui n'est pas dans une table traverse **intact**
(ajouter un conteneur d'état n'est pas neutre, #1664). `opaque_id` fabrique les
identifiants de substitution. `pattern_mining` agrège des signatures déjà
sanitisées en motifs corpus-level — sans jamais revoir le texte.

In [1]:
import copy
import json
import os
from pathlib import Path

EXAMPLES = json.loads(
    Path("docs/coursia_contrib/privacy_mining_examples.json").read_text(encoding="utf-8")
)
assert EXAMPLES["asset"] == "privacy_mining"
print("sections:", [k for k in EXAMPLES if k.endswith("_cases")])

# Sel SYNTHÉTIQUE pour tout le notebook — le vrai sel vit dans .env / CI.
os.environ["OPAQUE_ID_SALT"] = "synthetic-asset-salt"

from argumentation_analysis.evaluation.opaque_id import opaque_id
from argumentation_analysis.evaluation.sanitize_state import sanitize_state
from argumentation_analysis.evaluation import pattern_mining as pm

sections: ['opaque_cases', 'sanitize_cases', 'mining_cases']


## 1. `opaque_id` — déterministe, non réversible, et vérifiable si le sel fuit

`opaque_id(name, salt)` rend les 8 premiers hex de `sha256(salt + name)` :
même (nom, sel) → même ID ; deux noms distincts sous un sel → ID distincts.
La non-réversibilité ne protège rien seule : avec un sel **public**, qui
devine un nom candidat recalcul son ID en O(1) et **confirme** le match —
un oracle de confirmation, pas une obfuscation. D'où #1973 : plus de sel par
défaut dans le fichier, `RuntimeError` sinon.

In [2]:
stored = {c["name"]: c for c in EXAMPLES["opaque_cases"]}

s = stored["deterministic_and_distinct"]
print("Source A / s1 :", opaque_id("Source A", salt="s1"))
print("Source A / s1 :", opaque_id("Source A", salt="s1"), "<- stable")
print("Source B / s1 :", opaque_id("Source B", salt="s1"), "<- distinct")
print("Source A / s2 :", opaque_id("Source A", salt="s2"), "<- autre sel, tout change")
assert opaque_id("Source A", salt="s1") == s["id_a_1"] == s["id_a_2"]
assert opaque_id("Source B", salt="s1") == s["id_b_same_salt"] != s["id_a_1"]
assert opaque_id("Source A", salt="s2") == s["id_a_other_salt"] != s["id_a_1"]

saved = os.environ.pop("OPAQUE_ID_SALT")
try:
    opaque_id("Source A")
    raise SystemExit("devrait lever")
except RuntimeError as e:
    print("sans sel :", str(e)[:80], "…")
    assert "OPAQUE_ID_SALT is required" in str(e)
    assert stored["no_salt_raises"]["raises"].startswith("RuntimeError")
os.environ["OPAQUE_ID_SALT"] = saved

Source A / s1 : a295797e
Source A / s1 : a295797e <- stable
Source B / s1 : 5f602022 <- distinct
Source A / s2 : 193c2319 <- autre sel, tout change
sans sel : OPAQUE_ID_SALT is required: no salt argument was passed and the OPAQUE_ID_SALT e …


## 2. `sanitize_state` — l'allowlist qui prévue les agrégats

Huit passes numérotées (1 strip top-level, 2/2b opacification, 3/4/4b-4d
dict-of-dicts, 5/5b-5f list-of-dicts, 6 symboles, 7 narratif → longueur,
8 struct → compteurs). Le contrat : **tout contenu quantitatif ou topologique
survit, tout texte nominatif part**. La fixture ci-dessous couvre chaque passe
avec du texte français synthétique — et un conteneur inconnu.

In [3]:
stored = {c["name"]: c for c in EXAMPLES["sanitize_cases"]}

STATE = {
    "source_name": "Discours de synthèse (nom réel)",
    "author": "Auteur Réel",
    "url": "https://example.invalid/doc",
    "raw_text": "Texte brut long du document source.",
    "source_id": "doc_source_001",
    "source_metadata": {"genre": "discours", "title": "Titre réel du discours", "channel": "TV"},
    "identified_arguments": {"arg1": "Le premier argument en clair", "arg2": {"score": 3}},
    "identified_fallacies": {"f1": {"type": "ad_hominem", "justification": "cite le texte réel", "severity": 2}},
    "argument_quality_scores": {"arg1": {"overall": 0.8, "llm_assessment": "narratif LLM citant l'argument"}},
    "dung_frameworks": {"fw1": {
        "name": "dung_grounded",
        "arguments": ["le premier argument en clair", "le second en clair"],
        "attacks": [["le premier argument en clair", "le second en clair"]],
        "extensions": {"extensions": [["le premier argument en clair"]], "count": 1, "sizes": [1], "all_members": ["le premier argument en clair"]},
        "formalism_specific": {
            "contraries": {"atome_a": "contraire_a"},
            "attack_weights": [{"source": "atome_a", "target": "atome_b", "weight": 0.7}],
            "criterion": "generalized_specificity",
        },
    }},
    "counter_arguments": [{"strategy": "reductio", "counter_content": "contre-argument en clair", "score": 0.5}],
    "extracts": [{"content": "extrait en clair", "claim_id": "c1"}],
    "probabilistic_results": [{"method": "ml", "arguments": ["argument en clair"], "acceptance_probabilities": {"argument en clair": 0.42}}],
    "dialogue_results": [{"topic": "sujet en clair", "trace": [{"round": 1, "speaker": "S1", "argument": "texte en clair", "target": "autre texte"}]}],
    "debate_transcripts": [{"round": 1, "topic": "sujet débat", "exchanges": [{"point": "point en clair", "rebuttal": "réponse en clair", "scheme": "expert_opinion"}]}],
    "nl_to_logic_translations": [{"source": "s", "logic": "P->Q", "original_text": "texte original", "variables": {"p": "signification en clair de p"}}],
    "belief_revision_results": [{"method": "m", "original": ["croyance en clair"], "revised": [], "minimal_retraction": {"options": [["croyance en clair"]], "cardinality": 1}}],
    "narrative_synthesis": "Long texte narratif de synthèse.",
    "final_conclusion": "Conclusion finale narrative.",
    "stakes_and_stakeholders": {"stakes": ["enjeu en clair"], "stakeholders": ["acteur en clair"], "rhetorical_register": "délibératif"},
    "identified_metrics": {"n_args": 2, "score": 0.9},
    "unknown_new_container": [{"free": "texte d'un conteneur non classé"}],
}
original = copy.deepcopy(STATE)
san = sanitize_state(STATE)
print("sanitisé — clés :", len(san), "| source_name présente :", "source_name" in san)

sanitisé — clés : 18 | source_name présente : False


### 2.1 Passes 1-2b : top-level retiré, identifiants opacifiés

`source_name`/`author`/`url`/`raw_text` disparaissent ; `source_id` devient
8 hex ; `source_metadata` garde ses **clés** (quelles métadonnées étaient
présentes) et opacifie chaque **valeur**.

In [4]:
s1 = stored["top_level_stripped_and_source_opacified"]
assert "source_name" not in san and not s1["source_name_present"]
assert "author" not in san and not s1["author_present"]
assert "raw_text" not in san and not s1["raw_text_present"]
print("source_id :", san["source_id"])
assert san["source_id"] == s1["source_id_after"] and s1["source_id_is_hex"]
print("source_metadata :", san["source_metadata"])
assert list(san["source_metadata"].keys()) == s1["metadata_keys_after"]
assert list(san["source_metadata"].values()) == s1["metadata_values_after"]
print("-> clés structurelles gardées, valeurs nominatives opacifiées")

source_id : 3599baaa
source_metadata : {'genre': '5b656822', 'title': '8a9e44dd', 'channel': '6fc47b79'}
-> clés structurelles gardées, valeurs nominatives opacifiées


### 2.2 Passes 3-4 : texte strip vs structure gardée

Un argument **chaîne** devient `{"text_stripped": true}` ; un argument
**structuré** passe intact. Dans `identified_fallacies`, `type` (vocabulaire
fermé) et `severity` (numérique) survivent, `justification` (narratif citant
le texte) part — même verdict pour `llm_assessment`, 0 valeur quantitative.

In [5]:
s2 = stored["arguments_and_justifications"]
assert san["identified_arguments"]["arg1"] == s2["string_argument_becomes"] == {"text_stripped": True}
assert san["identified_arguments"]["arg2"] == s2["structured_argument_survives"] == {"score": 3}
assert san["identified_fallacies"]["f1"] == s2["fallacy_after"]
assert "justification" not in san["identified_fallacies"]["f1"]
assert "llm_assessment" not in san["argument_quality_scores"]["arg1"]
assert san["argument_quality_scores"]["arg1"]["overall"] == 0.8
print("fallacy f1 après :", san["identified_fallacies"]["f1"])

fallacy f1 après : {'type': 'ad_hominem', 'severity': 2}


### 2.3 Passes 4b-4d : la topologie Dung survit à l'opacification

`attacks` garde son arité (paires), `extensions.count`/`sizes` survivent,
les **membres** deviennent opaques — les agrégats quantitatifs en aval ne
bougent pas. Dans le sidecar `formalism_specific` : atomes source opacifiés,
`criterion` (vocabulaire fermé) et `weight` (numérique) gardés.

In [6]:
s3 = stored["topology_survives_opacification"]
fw = san["dung_frameworks"]["fw1"]
assert len(fw["attacks"][0]) == s3["attacks_arity"] == 2
assert fw["extensions"]["count"] == s3["extensions_count_kept"] == 1
assert fw["extensions"]["sizes"] == s3["extensions_sizes_kept"] == [1]
assert fw["name"] == s3["name_kept"] == "dung_grounded"
assert fw["formalism_specific"]["criterion"] == s3["criterion_kept"]
assert fw["formalism_specific"]["attack_weights"][0]["weight"] == s3["weight_kept"] == 0.7
print("attacks   :", fw["attacks"])
print("extensions:", fw["extensions"])
print("contraries:", fw["formalism_specific"]["contraries"])
assert "le premier argument en clair" not in json.dumps(san, ensure_ascii=False)
print("-> aucun texte en clair ne subsiste dans tout l'état sanitisé")

attacks   : [['d4073163', '0bbe670b']]
extensions: {'extensions': [['d4073163']], 'count': 1, 'sizes': [1], 'all_members': ['d4073163']}
contraries: {'f72b5fab': '0021a491'}
-> aucun texte en clair ne subsiste dans tout l'état sanitisé


### 2.4 Passes 5b-5e : les conteneurs List[Dict]

Même logique un niveau plus bas : `method` (fermé) gardé, `arguments`
opaques ; les **clés** de `acceptance_probabilities` sont le texte de
l'argument → opacifiées, les **valeurs** numériques gardées — la
distribution survit. `speaker`/`scheme` fermés gardés ; `cardinality` gardé.

In [7]:
s4 = stored["list_containers"]
assert san["counter_arguments"][0] == s4["counter_arguments_after"]
assert san["extracts"][0] == s4["extracts_after"]
assert san["probabilistic_results"][0]["method"] == s4["prob_method_kept"]
assert san["probabilistic_results"][0]["arguments"] == s4["prob_arguments_opaque"]
assert list(san["probabilistic_results"][0]["acceptance_probabilities"].keys()) == s4["prob_mapping_keys_opaque"]
assert list(san["probabilistic_results"][0]["acceptance_probabilities"].values()) == s4["prob_values_kept"] == [0.42]
assert san["dialogue_results"][0]["trace"][0]["speaker"] == s4["dialogue_speaker_kept"] == "S1"
assert san["debate_transcripts"][0]["exchanges"][0]["scheme"] == s4["debate_scheme_kept"]
assert san["belief_revision_results"][0]["minimal_retraction"]["cardinality"] == s4["br_cardinality_kept"] == 1
print("probabilistic :", san["probabilistic_results"][0])

probabilistic : {'method': 'ml', 'arguments': ['fb22732b'], 'acceptance_probabilities': {'fb22732b': 0.42}}


### 2.5 Passes 7-8 : narratif et struct

Le narratif devient `{length, stripped}` ; `stakes_and_stakeholders` devient
des compteurs + drapeaux de présence (`has_rhetorical_register`). Et le
piège design : un **conteneur inconnu traverse intact** — l'allowlist est
le contrat, pas un filet.

In [8]:
s5 = stored["narrative_and_struct_reductions"]
assert san["narrative_synthesis"] == s5["narrative_after"]
assert san["narrative_synthesis"]["length"] == s5["narrative_length"] == len("Long texte narratif de synthèse.")
assert san["stakes_and_stakeholders"] == s5["stakes_after"]
assert san["stakes_and_stakeholders"]["has_rhetorical_register"] is True

s6 = stored["unknown_container_traverses_intact"]
assert san["unknown_new_container"] == s6["unknown_container_after"]
assert san["identified_metrics"] == s6["metrics_untouched"] == {"n_args": 2, "score": 0.9}
print("conteneur inconnu après :", san["unknown_new_container"], "<- INTACT, par design")

s7 = stored["input_not_mutated"]
assert STATE == original and s7["original_intact"] is True
print("l'entrée n'est PAS mutée (deepcopy)")

s8 = stored["salt_requirement_is_conditional"]
assert sanitize_state({"identified_metrics": {"n": 1}}) == s8["state_without_opacifiable_fields"]
saved = os.environ.pop("OPAQUE_ID_SALT")
try:
    sanitize_state({"source_id": "doc_x"})
    raise SystemExit("devrait lever")
except RuntimeError:
    print("source_id sans sel -> RuntimeError (opaque_id appelé)")
os.environ["OPAQUE_ID_SALT"] = saved
print("sans champ opacifiable -> OK sans sel")

conteneur inconnu après : [{'free': "texte d'un conteneur non classé"}] <- INTACT, par design
l'entrée n'est PAS mutée (deepcopy)
source_id sans sel -> RuntimeError (opaque_id appelé)
sans champ opacifiable -> OK sans sel


## 3. `pattern_mining` — motifs corpus-level sur signatures sanitisées

Consomme des signatures `{metadata, state}` (les mêmes états, sanitisés) :
spectre de sophismes par cluster, asymétrie **Tricherie** (8 types,
auto-biais) ↔ **Influence** (12 types, procédé rhétorique), co-occurrence
avec support/confidence/lift/jaccard, couverture croisée informal × formel,
et trois détecteurs formels.

In [9]:
stored = {c["name"]: c for c in EXAMPLES["mining_cases"]}

SIGS = [
    {"metadata": {"cluster_id": "corpus_A"}, "state": {
        "identified_fallacies": {
            "f1": {"type": "ad_hominem", "source_arg": "arg1"},
            "f2": {"type": "straw_man", "source_arg": "arg1"},
            "f3": {"type": "cherry_picking", "source_arg": "arg2"},
        },
        "fol_analysis_results": [{"formula": "F", "valid": False}],
        "dung_frameworks": {"fw": {"attacks": [{"from": "a", "to": "b"}]}},
        "jtms_retraction_chain": [{"trigger": "x"}],
    }},
    {"metadata": {"cluster_id": "corpus_B"}, "state": {
        "identified_fallacies": {
            "f1": {"type": "appeal_to_authority", "source_arg": "arg1"},
            "f2": {"type": "false_cause", "source_arg": "arg2"},
        },
        "dung_frameworks": {"fw": {"attacks": [{"from": "a", "to": "b"}, {"from": "b", "to": "c"}]}},
    }},
]

spec = pm.fallacy_spectrum(SIGS)
print(json.dumps(spec, ensure_ascii=False, indent=1))
assert spec == stored["spectrum"]["result"]
total = sum(spec["corpus_A"].values())
print("somme des fréquences corpus_A :", total, "(arrondies à 4 décimales)")
# trois tiers arrondis à 4 décimales -> 0.9999, pas 1.0 exact : l'artefact est mesuré
assert abs(total - 1.0) < 2e-4

{
 "corpus_A": {
  "ad_hominem": 0.3333,
  "straw_man": 0.3333,
  "cherry_picking": 0.3333
 },
 "corpus_B": {
  "appeal_to_authority": 0.5,
  "false_cause": 0.5
 }
}
somme des fréquences corpus_A : 0.9999 (arrondies à 4 décimales)


In [10]:
tvi = pm.trick_vs_influence_ratio(SIGS)
print(json.dumps(tvi, ensure_ascii=False, indent=1))
assert tvi == stored["trick_vs_influence"]["result"]
# corpus_A : 2 influence (ad_hominem, straw_man) vs 1 tricherie (cherry_picking)
assert tvi["corpus_A"]["asymmetry"] == round((2 - 1) / (2 + 1), 4)

edge = stored["trick_vs_influence_edges"]
only_inf = pm.trick_vs_influence_ratio([
    {"metadata": {"cluster_id": "only_influence"},
     "state": {"identified_fallacies": {"f": {"type": "bandwagon", "source_arg": "a"}}}}
])
print("tricherie=0 :", only_inf["only_influence"])
assert only_inf == edge["only_influence"]
assert only_inf["only_influence"]["ratio"] == 1e9  # plafonné, pas de division par zéro

{
 "corpus_A": {
  "tricherie_share": 0.3333,
  "influence_share": 0.6667,
  "ratio": 2.0,
  "asymmetry": 0.3333
 },
 "corpus_B": {
  "tricherie_share": 0.5,
  "influence_share": 0.5,
  "ratio": 1.0,
  "asymmetry": 0.0
 }
}
tricherie=0 : {'tricherie_share': 0.0, 'influence_share': 1.0, 'ratio': 1000000000.0, 'asymmetry': 1.0}


In [11]:
co_arg = pm.cooccurrence_matrix(SIGS, unit="argument")
print(json.dumps(co_arg, ensure_ascii=False, indent=1))
assert co_arg == stored["cooccurrence_argument_unit"]["result"]
# unité argument : arg1 de corpus_A porte ad_hominem + straw_man -> paire co-occurrente
assert any(p["a"] == "ad_hominem" and p["b"] == "straw_man" for p in co_arg["pairs"])
top = co_arg["pairs"][0]
print("top paire par lift :", top["a"], "+", top["b"], "lift =", top["lift"])

{
 "pairs": [
  {
   "a": "ad_hominem",
   "b": "straw_man",
   "support": 1,
   "confidence": 1.0,
   "lift": 4.0,
   "jaccard": 1.0
  }
 ],
 "unit_count": 4
}
top paire par lift : ad_hominem + straw_man lift = 4.0


In [12]:
cc = pm.cross_coverage(SIGS)
assert cc == stored["cross_coverage"]["result"]
print("corpus_B appeal_to_authority :", json.dumps(cc["appeal_to_authority"]))
# corpus_B : pas de FOL, pas de JTMS, mais c est attaqué sans attaquer -> unsupported
assert cc["appeal_to_authority"]["per_signature_rate"]["dung_unsupported"] == 1.0
assert cc["appeal_to_authority"]["per_signature_rate"]["fol_invalid"] == 0.0
print("-> per_signature compte chaque signature une fois ; per_occurrence chaque occurrence")

corpus_B appeal_to_authority : {"per_signature_rate": {"fol_invalid": 0.0, "dung_unsupported": 1.0, "jtms_retraction": 0.0}, "per_occurrence_rate": {"fol_invalid": 0.0, "dung_unsupported": 1.0, "jtms_retraction": 0.0}}
-> per_signature compte chaque signature une fois ; per_occurrence chaque occurrence


### 3.1 Les trois détecteurs formels — et leurs pièges mesurés

`dung_topology` : densité **directed** `n(n-1)` et aplatissement des
extensions. Piège mesuré : avec des attactions en forme de dicts
(`{from, to}`), `n_attacks` compte des **entrées**, pas des arêtes —
`density` lit 1.0. `atms_branching` : contrat tri-états #1650 —
`coherent=False` → contradiction, `coherent` **absent** → non classé
(exclu et warn, jamais fabriqué par `not coherent`). Un détecteur qui lève
rend `{"error": 1.0}`, jamais un crash.

In [13]:
det = pm.run_formal_detectors(SIGS[0])
assert det == stored["formal_detectors"]["result"]
print("sur SIGS[0] (attaques en dicts) :", json.dumps(det["dung_topology"]))
print("-> n_attacks=1 (UNE entrée dict, pas 1 arête dirigée), density=1.0 : piège mesuré")

topo_sig = {"state": {"dung_frameworks": {"fw": {
    "arguments": ["a", "b", "c"],
    "attacks": [["a", "b"], ["b", "c"]],
    "extensions": {"grounded": ["a", "c"], "preferred": [["a", "c"]]},
}}}}
topo = pm.DungTopologyDetector().detect(topo_sig)
print("arêtes en paires :", json.dumps(topo))
assert topo == stored["dung_topology_directed_density"]["result"]
assert topo["density"] == stored["dung_topology_directed_density"]["density_value"] == round(2 / 6, 4)
print("-> 2 arêtes / (3*2) directed =", topo["density"], "(pas n(n-1)/2)")

import logging
logging.disable(logging.WARNING)
atms = pm.AtmsBranchingDetector().detect({"state": {"atms_contexts": [
    {"assumptions": ["p", "q"], "coherent": True},
    {"assumptions": ["r"], "coherent": False},
    {"assumptions": []},  # pas de clé -> non classé, PAS contradiction
]}})
logging.disable(logging.NOTSET)
print("atms 3 états :", json.dumps(atms))
assert atms == stored["atms_coherent_three_state_1650"]["result"]
assert atms["contradiction_rate"] == round(1 / 3, 4)  # 1/3, le non-classé exclu

class Boom:
    name = "boom"
    def detect(self, signature):
        raise ValueError("kaboom")

res = pm.run_formal_detectors({}, detectors=[Boom()])
assert res == stored["detector_exception_is_error_flag"]["result"] == {"boom": {"error": 1.0}}
print("détecteur qui lève :", res, "<- flag, pas de crash")

sur SIGS[0] (attaques en dicts) : {"n_args": 0.0, "n_attacks": 1.0, "density": 1.0, "n_extensions": 0.0, "max_extension_size": 0.0}
-> n_attacks=1 (UNE entrée dict, pas 1 arête dirigée), density=1.0 : piège mesuré
arêtes en paires : {"n_args": 3.0, "n_attacks": 2.0, "density": 0.3333, "n_extensions": 2.0, "max_extension_size": 2.0}
-> 2 arêtes / (3*2) directed = 0.3333 (pas n(n-1)/2)
atms 3 états : {"max_assumption_count": 2.0, "avg_assumptions": 1.0, "contradiction_rate": 0.3333}
détecteur qui lève : {'boom': {'error': 1.0}} <- flag, pas de crash


## Récapitulatif

| Étape | Contrat | Piège mesuré |
|---|---|---|
| `opaque_id` | déterministe, distinct | sel public = oracle de confirmation (#1973 : RuntimeError sans sel) |
| `sanitize_state` | agrégats surviennent, texte part | conteneur inconnu traverse INTACT (allowlist = le contrat, #1664) |
| `pattern_mining` | motifs sur signatures sanitises | attaques-dicts → n_attacks compte des entrées ; #1650 tri-états coherent |

Corpus-free par construction : fixtures synthétiques françaises, aucun
identifiant réel, sel synthétique visible. Zéro LLM, zéro JVM.